In [1]:
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification, 
                          TrainingArguments, 
                          Trainer)
from datasets import load_dataset

from pathlib import Path
import numpy as np
import torch
print(torch.cuda.get_device_name(0))

NVIDIA GeForce GTX 1650 with Max-Q Design


In [2]:
data_dir = Path("../data/processed/")
ckpt = "distilbert-base-uncased"

In [3]:
data_files = {
    "train": str(data_dir/"wndp-api-data-train.parquet"),
    "val": str(data_dir/"wndp-api-data-val.parquet"),
    "test": str(data_dir/"wndp-api-data-test.parquet"),
}

ds = load_dataset("parquet", data_files=data_files)
ds.set_format("torch")
ds

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 10632
    })
    val: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2658
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2346
    })
})

In [4]:
labels = [
    'clinically_healthy',
    'dermatologic_disease',
    'gastrointestinal_disease',
    'hematologic_disease',
    'neurologic_disease',
    'nonspecific',
    'nutritional_disease',
    'ocular_disease',
    'physical_injury',
    'respiratory_disease',
    'urogenital_disease'
]
id2label = {idx:label for idx,label in enumerate(labels)}
label2id = {label:idx for idx,label in enumerate(labels)}

In [5]:
num_labels = len(ds["train"][0]["labels"])
tokenizer = AutoTokenizer.from_pretrained(ckpt, use_fast=True)

In [6]:
sample = ds["train"][0]
sample.keys()

dict_keys(['labels', 'input_ids', 'attention_mask'])

In [7]:
tokenizer.decode(sample["input_ids"])

'[CLS] no patient info. too young [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]'

In [8]:
sample["labels"]

tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [9]:
[id2label[idx] for idx, label in enumerate(sample['labels']) if label == 1.0]

['clinically_healthy']

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
            ckpt,
            num_labels=num_labels,
            problem_type="multi_label_classification",
            id2label=id2label,
            label2id=label2id
        )

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['pre_classifier.bias', 'pre_classifier.weight', 'classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
batch_size = 64
metric_name = "f1"

In [12]:
args = TrainingArguments(
    f"wndp-exp",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    num_train_epochs=10,
    weight_decay=1e-2,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    load_best_model_at_end=True,
    metric_for_best_model=metric_name
)

In [13]:
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from transformers import EvalPrediction
import torch
    
# source: https://jesusleal.io/2021/04/21/Longformer-multilabel-classification/
def multi_label_metrics(predictions, labels, threshold=0.5):
    # first, apply sigmoid on predictions which are of shape (batch_size, num_labels)
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions))
    # next, use threshold to turn them into integer predictions
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= threshold)] = 1
    # finally, compute metrics
    y_true = labels
    f1_micro_average = f1_score(y_true=y_true, y_pred=y_pred, average='micro')
    roc_auc = roc_auc_score(y_true, y_pred, average = 'micro')
    accuracy = accuracy_score(y_true, y_pred)
    # return as dictionary
    metrics = {'f1': f1_micro_average,
               'roc_auc': roc_auc,
               'accuracy': accuracy}
    return metrics

def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions, 
            tuple) else p.predictions
    result = multi_label_metrics(
        predictions=preds, 
        labels=p.label_ids)
    return result

In [14]:
ds["train"][0]["labels"]

tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [15]:
ds["train"]["input_ids"][0]

tensor([  101,  2053,  5776, 18558,  1012,  2205,  2402,   102,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0])

In [16]:
outputs = model(
            input_ids=ds["train"]["input_ids"][0].unsqueeze(0),
            labels=ds["train"][0]["labels"].unsqueeze(0)            
)
outputs.logits

We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


tensor([[ 0.0263, -0.0406, -0.1653, -0.0563, -0.0324, -0.0425,  0.0691,  0.0105,
         -0.0065,  0.0603, -0.1581]], grad_fn=<AddmmBackward0>)

In [17]:
trainer = Trainer(
    model,
    args,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [18]:
%%time
trainer.train()

You're using a DistilBertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,F1,Roc Auc,Accuracy
1,No log,0.133108,0.792309,0.858807,0.635816
2,No log,0.107298,0.838305,0.904813,0.691497
3,0.142700,0.104402,0.842213,0.898569,0.704289
4,0.142700,0.109665,0.846076,0.906180,0.715199
5,0.142700,0.111240,0.848953,0.916182,0.719714
6,0.044400,0.117822,0.848233,0.909627,0.724605
7,0.044400,0.124516,0.852599,0.914585,0.726110
8,0.044400,0.128777,0.853965,0.913882,0.729872
9,0.015900,0.131218,0.856573,0.915452,0.731377
10,0.015900,0.132873,0.857750,0.918294,0.735515


CPU times: total: 1h 39min 51s
Wall time: 3h 11min 58s


TrainOutput(global_step=1670, training_loss=0.06176705574561022, metrics={'train_runtime': 11518.218, 'train_samples_per_second': 9.231, 'train_steps_per_second': 0.145, 'total_flos': 3521548581949440.0, 'train_loss': 0.06176705574561022, 'epoch': 10.0})

In [19]:
trainer.evaluate()

{'eval_loss': 0.13287319242954254,
 'eval_f1': 0.8577500708415982,
 'eval_roc_auc': 0.9182944832494808,
 'eval_accuracy': 0.735515425131678,
 'eval_runtime': 41.0002,
 'eval_samples_per_second': 64.829,
 'eval_steps_per_second': 1.024,
 'epoch': 10.0}

In [20]:
sample = "found on the ground by window - breathing hard, eyes not open, couldn't stand up, ants covering him, some spazmotic movements of leg, wing, seemed better today. emaciated fledgling with torticollis. Neurologic: torticollis Legs / Feet / Hocks: not using legs. poor prognosis given age, emaciation, and degree of debilitation"

### example text

sample = "found on the ground by window - breathing hard, eyes not open, couldn't stand up, ants covering him, some spazmotic movements of leg, wing, seemed better today. emaciated fledgling with torticollis. Neurologic: torticollis Legs / Feet / Hocks: not using legs. poor prognosis given age, emaciation, and degree of debilitation"

In [21]:
enc = tokenizer(sample, return_tensors="pt")

In [22]:
enc = {k: v.to(trainer.model.device) for k,v in enc.items()}

In [23]:
outputs = trainer.model(**enc)

In [24]:
outputs

SequenceClassifierOutput(loss=None, logits=tensor([[-8.2980, -7.5297, -5.3929, -6.9765,  5.5375, -7.7283,  4.5436, -7.4444,
         -6.1811, -6.4983, -8.4218]], device='cuda:0',
       grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [25]:
import torch.nn.functional as F

In [26]:
probs = F.sigmoid(outputs.logits.squeeze().detach().cpu())

In [27]:
probs

tensor([2.4896e-04, 5.3659e-04, 4.5282e-03, 9.3273e-04, 9.9608e-01, 4.4000e-04,
        9.8948e-01, 5.8435e-04, 2.0638e-03, 1.5037e-03, 2.1998e-04])

In [28]:
preds = (probs > 0.5).int()

In [29]:
predicted_labels = [id2label[idx] for idx, label in enumerate(preds) if label == 1.0]

In [30]:
predicted_labels

['neurologic_disease', 'nutritional_disease']